In [14]:
import json
import os

import pandas as pd

# --- Local (with a .env file) ---
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]


# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [15]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is land breeze?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

A land breeze is a type of local wind that occurs when the land cools faster than the sea or a large body of water. This typically happens at night or during the early morning hours.

Here's how it works:

1. During the day, the land heats up faster than the water due to its lower specific heat capacity.
2. As the sun sets, the land starts to cool down rapidly, while the water retains its heat longer.
3. As the land cools, the air above it also cools and becomes denser.
4. This denser air then flows towards the sea or the warmer area, creating a gentle wind that blows from the land to the sea.

Land breezes are usually gentle and can be quite refreshing, especially in coastal areas. They can also play a role in shaping local weather patterns and influencing the formation of clouds and precipitation.

The opposite of a land breeze is a sea breeze, which occurs when the land heats up faster than the sea during the day, causing the air to rise and create a wind that blows from the sea to 

## Student Reasoning — Anatomy of a Call

1. Difference between system and user

System: Provides the AI with instructions about how it should behave, respond, or what role it should perform.
Example: “Act as a helpful mathematics tutor.”
User: Contains the specific question, task, or instruction that the AI needs to answer.
Example: “Explain eigenvalues and eigenvectors in simple terms.”

2. What is a token?

A token is a small unit of text that a large language model (LLM) reads and processes. It can be a complete word, part of a word, or sometimes a character.

Why do API providers charge based on tokens instead of requests?

API providers use tokens for billing because each request can contain and produce different amounts of text. Counting tokens gives a more accurate way to measure how much information the model processes and generates.

In [16]:
# TODO: Ask the SAME question 5 times at temperature=0.0
# and 5 times at temperature=1.2.

# TODO: Print all 10 answers, grouped by temperature.
question = "What is reverse osmosis?"

print("--- Temperature 0.0 ---")
for i in range(5):
  answer = ask_llm(question, temperature=0.0)
  print(f"{i + 1}. {answer.choices[0].message.content}")

print("\n--- Temperature 1.2 ---")
for i in range(5):
  answer = ask_llm(question, temperature=1.2)
  print(f"{i + 1}. {answer.choices[0].message.content}")

--- Temperature 0.0 ---
1. Reverse osmosis (RO) is a water purification process that uses a semi-permeable membrane to remove impurities and contaminants from water. The process involves applying pressure to force the water through the membrane, which has tiny pores that allow water molecules to pass through while blocking larger particles and impurities.

Here's a step-by-step explanation of the reverse osmosis process:

1. **Pre-treatment**: The water is pre-treated to remove larger particles and debris that could damage the membrane.
2. **Pressurization**: The pre-treated water is pressurized to force it through the semi-permeable membrane.
3. **Filtration**: The pressurized water is forced through the membrane, which has pores that are typically 0.0001 microns in size. This size is small enough to block most impurities, including:
	* Dissolved solids (e.g., salt, minerals)
	* Bacteria
	* Viruses
	* Heavy metals
	* Pesticides
	* Herbicides
4. **Separation**: The water molecules pass

## Student Reasoning — Temperature
i Temperature = 0.0: The responses were highly consistent and had very similar wording, structure, and ideas when explaining the main concepts of banking systems.
Temperature = 1.2: The responses showed greater variation. The model used different wording, examples, and response structures. In one case, the response also reached the max_tokens limit.

ii For a loan decision-support system, a low temperature (approximately 0.0–0.3) would be more suitable because the system needs to produce consistent and predictable decisions with minimal randomness.

In [17]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [18]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
SUMMARY_PROMPT_V1 = "Summarize this:"

#   Run it on L002 and L006. Read the output critically.
for letter_id in ["L002", "L006"]:
    response = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    )
    print(f"\n--- {letter_id} ---")
    print(response.choices[0].message.content)


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications factually and neutrally.
Do not invent or assume any details.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
for letter_id in ["L002", "L006"]:
    response = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )
    print(f"\n--- {letter_id} ---")
    print(response.choices[0].message.content)
    # TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for letter_id in ["L002", "L006"]:
    v1 = ask_llm(
        f"{SUMMARY_PROMPT_V1}\n\n{LETTERS[letter_id]}"
    )
    
    v2 = ask_llm(
        SUMMARY_PROMPT_V2.format(letter_text=LETTERS[letter_id]),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )

    print(f"\n========== {letter_id} ==========")
    print("\nV1:")
    print(v1.choices[0].message.content)
    
    print("\nV2:")
    print(v2.choices[0].message.content)


--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances recover. However, he currently has no collateral to offer.

--- L006 ---
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He claims to be "business-minded" based on his friends' opinions, but has no prior experience. He promises to repay the loan in one year, once the businesses are successful, and offers his trustworthiness as assurance, as he has no collateral to provide.

--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has 

## Student Reasoning — Summarization Prompts
1. Issues with V1 and how V2 improved them

V1 sometimes included information that was not actually provided. For example, in L006, V1 stated that Kofi had “no prior experience,” even though the letter only mentioned that he had not yet started the businesses. V1 also described Kofi’s trustworthiness as “assurance,” which was not directly mentioned in the original text. V2 improved this by staying factual and neutral, avoiding unsupported assumptions, and following the required 3–4 sentence limit.

2. Why is “no invented details” important?

Loan decisions should rely only on accurate information provided by the applicant. Adding information that is not true or supported could unfairly influence the outcome of someone's loan application.

This type of error is known as hallucination in large language models (LLMs), where the model generates information that is unsupported or incorrect.